# Heatmap $|\psi_{\rm out}(\omega_a,\omega_b)|$ para $N$ sítios cross-Kerr

Este notebook explica, expressão por expressão, tudo o que entra no mapa de calor gerado por
`heatmap_omegaa_omegab_n_sitios.py`: a **amplitude espectral do par de fótons espalhado**

$$
\psi_{\rm out}(\omega_a,\omega_b)=\langle\omega_a\,\omega_b|\,S\,|\xi\xi\rangle
$$

no plano das frequências de **saída** $(\omega_a,\omega_b)$, para uma cadeia de **$N$ sítios cross-Kerr idênticos**
atravessada por dois fótons **contrapropagantes** que entram no mesmo pacote gaussiano $\xi$
(centro $\omega_c$, largura $\sigma$).

Ele generaliza `heatmap_omegaa_omegab.py` ($N=2$) e usa a mesma matriz S de
`fidelidade_sigma/calcular_sigma_n_sitios.py`. Enquanto aquele cálculo reduz tudo a um único número
(a fidelidade $F_1$), aqui vemos **onde, no espaço de frequências, a amplitude de saída está**.

**Roteiro**

| Seção | O que é explicado | Onde aparece no código |
|---|---|---|
| 1 | Parâmetros físicos, numéricos e da figura | bloco de parâmetros no topo do script |
| 2 | Blocos básicos: $\Gamma$, $t$, $a$, $S_1$, $\xi$ | `Gamma`, `t_site`, `a_site`, `S1`, `xi_in` |
| 3 | Matriz S de $N$ sítios e o núcleo $K(E)$ | fórmula da docstring; `K` em `psi_out` |
| 4 | $\psi_{\rm out}$ e a redução a convoluções $P_j(E)$ | `psi_out` |
| 5 | Correção da fase de fóton único | `aplicar_correcao` |
| 6 | Grade numérica das convoluções | `grade` |
| 7 | Verificações: integral direta, unitariedade, ligação com $F_1$ | — |
| 8 | O gráfico, elemento por elemento | `main`, `cores`, `FIELD_LABELS` |
| 9 | Os outros campos: `abs_nl`, `real`, `imag`, `phase` | `field` |
| 10 | Dependência em $N$ e em $L$ | — |

In [ ]:
import os

import numpy as np
import matplotlib.pyplot as plt
from matplotlib.colors import LinearSegmentedColormap
from scipy.integrate import quad
from scipy.signal import fftconvolve

---
## 1. Parâmetros

### 1.1 Parâmetros físicos

Todos os $N$ sítios são **idênticos**; cada um é um ressonador/emissor com

* **largura** (taxa de decaimento) $\gamma$,
* **dessintonia** $\Delta$,
* **acoplamento não linear** (cross-Kerr) $\chi$.

Sítios vizinhos estão separados por uma distância $L$ (em unidades em que a velocidade de propagação vale 1,
de modo que um fóton de frequência $\omega$ acumula a fase $e^{i\omega L}$ ao ir de um sítio ao próximo).

O pacote de entrada é gaussiano, de largura $\sigma$, centrado em

$$
\omega_c=\Delta+\omega_0 ,
$$

em que $\omega_0$ é a frequência central **relativa a $\Delta$** (`omega_c` é derivado, não deve ser editado).
Todas as frequências estão em unidades de $\gamma$ ($\gamma=1$) e $L$ em unidades de $1/\gamma$.

| Variável | Símbolo | Valor no script | Significado |
|---|---|---|---|
| `N` | $N$ | `10` | número de sítios |
| `gamma` | $\gamma$ | `1.0` | largura de cada sítio |
| `chi` | $\chi$ | `1000.0` | acoplamento não linear de cada sítio |
| `Delta` | $\Delta$ | `0.0` | dessintonia de cada sítio |
| `omega0` | $\omega_0$ | `0.0` | centro do pacote relativo a $\Delta$ |
| `L` | $L$ | `0` | separação entre sítios vizinhos |
| `sigma` | $\sigma$ | `1.0` | largura espectral do pacote |

### 1.2 Plano $(\omega_a,\omega_b)$

| Variável | Valor | Significado |
|---|---|---|
| `omega_min`, `omega_max` | `-3.0`, `3.0` | limites dos **dois** eixos (figura quadrada) |
| `n_omega` | `2000` | pontos por eixo: o mapa tem $n_\omega^2=4\times10^6$ pixels |

### 1.3 Parâmetros numéricos das convoluções (seção 6)

| Variável | Valor | Significado |
|---|---|---|
| `k_sigma` | `9.0` | meia-janela de integração, em unidades de $\sigma$ |
| `pts_per_width` | `14` | pontos por menor escala, $\min(\sigma,\gamma,1/(N\lvert L\rvert))$ |
| `n_min` | `2001` | número mínimo de pontos da grade |
| `n_max` | `600001` | teto de pontos (proteção de tempo/memória) |

### 1.4 O que colorir (seções 5, 8 e 9)

| Variável | Valor | Significado |
|---|---|---|
| `field` | `'abs'` | `'abs'`, `'abs_nl'`, `'real'`, `'imag'` ou `'phase'` |
| `aplicar_correcao` | `True` | remove a fase de fóton único (não altera `'abs'`/`'abs_nl'`) |
| `vmin`, `vmax` | `0.0`, `None` | faixa da barra de cores |

Os parâmetros de cor e de figura (`cores`, `cmap`, `outfile`, `dpi`, ...) estão na seção 8.

Abaixo, os mesmos parâmetros ficam reunidos numa classe, para podermos trocar $N$, $L$, $\sigma$... sem
editar variáveis globais. **Único valor diferente do script:** `n_omega = 500`, para o notebook rodar rápido
(o script usa 2000, que só deixa a imagem mais fina).

In [ ]:
class Params:
    # Parâmetros físicos (mesmos valores de heatmap_omegaa_omegab_n_sitios.py)
    N = 10             # número de sítios
    gamma = 1.0        # taxa de decaimento (igual em todos os sítios)
    chi = 1000.0       # acoplamento não linear (igual em todos os sítios)
    Delta = 0.0        # dessintonia (igual em todos os sítios)
    omega0 = 0.0       # frequência central do pacote, relativa a Delta
    L = 0.0            # separação entre sítios vizinhos
    sigma = 1.0        # largura do pacote de entrada

    # Plano (wa, wb)
    omega_min = -3.0
    omega_max = 3.0
    n_omega = 500      # o script usa 2000

    # Parâmetros numéricos das convoluções P_j(E) (seção 6)
    k_sigma = 9.0
    pts_per_width = 14
    n_min = 2001
    n_max = 600001

    # O que colorir
    aplicar_correcao = True

    def __init__(self, **kw):
        for k, v in kw.items():
            if not hasattr(self, k):
                raise AttributeError(k)
            setattr(self, k, v)

    @property
    def omega_c(self):
        return self.Delta + self.omega0      # frequência central efetiva


P = Params()

---
## 2. Blocos básicos

### 2.1 Denominador de Breit–Wigner $\Gamma(\omega)$

$$
\boxed{\;\Gamma(\omega)=\frac{\gamma}{2}+i\,(\Delta-\omega)\;}
$$

É o denominador da resposta linear de um sítio: $\Gamma(\omega)=0$ em $\omega=\Delta-i\gamma/2$, o polo
lorentziano de largura $\gamma$. No código: `Gamma(w)`.

### 2.2 Fase de fóton único de um sítio $t(\omega)$

Um único fóton que atravessa um sítio só ganha uma fase (não há perda):

$$
\boxed{\;t(\omega)=\frac{\Gamma^{*}(\omega)}{\Gamma(\omega)}
=\frac{\gamma/2-i(\Delta-\omega)}{\gamma/2+i(\Delta-\omega)}\;},\qquad |t(\omega)|=1 .
$$

Na ressonância ($\omega=\Delta$), $\Gamma=\gamma/2$ é real e $t=1$; longe dela ($|\omega-\Delta|\gg\gamma$),
$t\to-1$. A fase de $t$ dá uma volta completa ($2\pi$) ao atravessar a ressonância, quase toda numa janela
de largura $\sim\gamma$. No código: `t_site(w)`.

### 2.3 Sítio + propagação: $a(\omega)$

A cada sítio segue-se um trecho de propagação $e^{i\omega L}$ até o próximo. O "tijolo" da cadeia é

$$
\boxed{\;a(\omega)=t(\omega)\,e^{i\omega L}\;},\qquad |a(\omega)|=1 .
$$

No código: `a_site(w)`. É com potências de $a$ que se escrevem os termos não lineares (seção 3).

### 2.4 Amplitude de fóton único da cadeia inteira $S_1(\omega)$

Um fóton sozinho atravessa $N$ sítios e $N-1$ trechos de propagação:

$$
\langle\omega^-|\nu^+\rangle=S_1(\omega)\,\delta(\omega-\nu),\qquad
\boxed{\;S_1(\omega)=e^{i(N-1)\omega L}\,t(\omega)^{N}\;}=a(\omega)^{N}\,e^{-i\omega L}.
$$

O expoente de $e^{i\omega L}$ é $N-1$, não $N$: depois do último sítio não há mais propagação a contabilizar.
No código: `S1(w)`.

### 2.5 Pacote de onda gaussiano $\xi(\omega)$

$$
\boxed{\;\xi(\omega)=\left(\frac{1}{2\pi\sigma^{2}}\right)^{1/4}
\exp\!\left[-\frac{(\omega-\omega_c)^{2}}{4\sigma^{2}}\right]\;}
$$

Gaussiana em amplitude, com largura de **intensidade** $|\xi|^2$ igual a $\sigma$ (desvio padrão), normalizada:
$\int|\xi(\omega)|^2d\omega=1$. Os dois fótons entram com o mesmo perfil, e o estado de entrada é

$$
|\xi\xi\rangle=\iint d\nu_a\,d\nu_b\;\xi(\nu_a)\,\xi(\nu_b)\,|\nu_a\,\nu_b\rangle ,
$$

cuja amplitude $\xi(\nu_a)\xi(\nu_b)$ tem máximo $1/(\sqrt{2\pi}\,\sigma)\approx0.399$ (para $\sigma=1$): um valor de
referência útil para ler a barra de cores do heatmap. No código: `xi_in(w)`.

In [ ]:
def Gamma(w, p=P):
    return p.gamma / 2 + 1j * (p.Delta - w)


def t_site(w, p=P):
    """Fase de espalhamento de um único sítio (|t|=1)."""
    return np.conj(Gamma(w, p)) / Gamma(w, p)


def a_site(w, p=P):
    """Fase de um sítio mais a propagação até o próximo: t(w) e^{iwL}."""
    return t_site(w, p) * np.exp(1j * w * p.L)


def S1(w, p=P):
    """Amplitude completa de um fóton: N sítios mais a propagação entre eles."""
    return t_site(w, p) ** p.N * np.exp(1j * (p.N - 1) * w * p.L)


def xi_in(w, p=P):
    """Pacote gaussiano normalizado, Eq. (4)."""
    return (1.0 / (2 * np.pi * p.sigma ** 2)) ** 0.25 * np.exp(-(w - p.omega_c) ** 2 / (4 * p.sigma ** 2))


# --- checagens rápidas das propriedades enunciadas acima ---
w = np.linspace(-50, 50, 20001)
Pt = Params(L=0.7)
print("|t| = 1, |a| = 1, |S1| = 1 :", np.allclose(np.abs(t_site(w)), 1), np.allclose(np.abs(a_site(w, Pt)), 1),
      np.allclose(np.abs(S1(w, Pt)), 1))
print("t(Delta) = 1               :", t_site(P.Delta))
print("t(Delta ± 1e4)             :", t_site(P.Delta + 1e4), t_site(P.Delta - 1e4))
print("S1 = a^N e^{-iwL}          :", np.allclose(S1(w, Pt), a_site(w, Pt) ** Pt.N * np.exp(-1j * w * Pt.L)))
print("∫|ξ|² dω = 1               :", np.trapezoid(np.abs(xi_in(w)) ** 2, w))
print("max ξ(wa)ξ(wb)             :", xi_in(P.omega_c) ** 2, "= 1/(sqrt(2π)σ) =", 1 / (np.sqrt(2 * np.pi) * P.sigma))

---
## 3. Matriz S de dois fótons para $N$ sítios

Os fótons entram com frequências $(\nu_a,\nu_b)$ e saem com $(\omega_a,\omega_b)$. No setor de dois fótons
contrapropagantes, a matriz S de $N$ sítios idênticos é

$$
\boxed{\;
S=\langle\omega_a^-|\nu_a^+\rangle\langle\omega_b^-|\nu_b^+\rangle
\;-\;\frac{i\chi\gamma^{2}}{\pi}\,
\frac{K(\nu_a,\nu_b)\;\delta(\omega_a+\omega_b-\nu_a-\nu_b)}
{\Gamma(\nu_b)\,\Gamma(\nu_a)\,\Gamma(\omega_b)\,\Gamma(\omega_a)}
\sum_{j=1}^{N}A^{N-j}B^{j-1}\;}
$$

com

$$
A=a(\omega_a)\,a(\nu_b)=\frac{\Gamma^*(\omega_a)\Gamma^*(\nu_b)}{\Gamma(\omega_a)\Gamma(\nu_b)}\,e^{i(\omega_a+\nu_b)L},
\qquad
B=a(\omega_b)\,a(\nu_a)=\frac{\Gamma^*(\omega_b)\Gamma^*(\nu_a)}{\Gamma(\omega_b)\Gamma(\nu_a)}\,e^{i(\omega_b+\nu_a)L}.
$$

**Leitura de cada pedaço.**

* **Primeiro termo (linear):** $S_1(\omega_a)S_1(\omega_b)\,\delta(\omega_a-\nu_a)\,\delta(\omega_b-\nu_b)$. Cada
  fóton atravessa a cadeia sozinho e só ganha a fase $S_1$; as frequências não mudam.
* **Segundo termo (não linear):** a interação entre os fótons. O índice $j$ da soma é o **sítio onde a interação
  acontece**. Como os fótons andam em sentidos opostos, um deles passa por $j-1$ sítios antes de chegar ao sítio
  $j$ e o outro por $N-j$; os fatores $A^{N-j}$ e $B^{j-1}$ são exatamente as fases lineares acumuladas nesses
  sítios "de passagem" (antes com a frequência de entrada $\nu$, depois com a de saída $\omega$).
* **A delta** $\delta(\omega_a+\omega_b-\nu_a-\nu_b)$ é a **conservação da energia total**: a interação pode
  redistribuir frequência entre os fótons, mas a soma é conservada. É ela que faz o estado de saída "vazar" para
  fora do pacote de entrada.
* **Os denominadores** $\Gamma(\cdot)$ limitam a interação a frequências dentro de $\sim\gamma$ da ressonância.

Para $N=2$ a soma tem dois termos ($j=1$: $A$; $j=2$: $B$), que são os `term2_bracket` e `term3_bracket`
de `fidelidade_sigma/calcular.py`.

### 3.1 O núcleo não linear $K$

$$
\boxed{\;K(\nu_a,\nu_b)=\left[1+\frac{2i\chi}{\Gamma(\nu_a)+\Gamma(\nu_b)}\right]^{-1}\;}
$$

Como $\Gamma(\nu_a)+\Gamma(\nu_b)=\gamma+i(2\Delta-E)$, com a **energia total** $E=\nu_a+\nu_b=\omega_a+\omega_b$,

$$
K=K(E)=\left[1+\frac{2i\chi}{\gamma+i(2\Delta-E)}\right]^{-1},
$$

que depende **só de $E$**. É isso que permite reduzir o cálculo a convoluções na seção 4.
No código: `K = 1.0 / (1.0 + 2j * chi / (gamma + 1j * (2 * Delta - E)))`.

**Limite saturado.** O que aparece em $S$ é o produto $\chi K$:

$$
\chi K(E)=\frac{\chi\,[\gamma+i(2\Delta-E)]}{\gamma+i(2\Delta-E)+2i\chi}
\;\xrightarrow{\;\chi\to\infty\;}\;
\frac{\gamma+i(2\Delta-E)}{2i}.
$$

O termo não linear **satura**: aumentar $\chi$ além de algumas vezes $\gamma$ quase não muda nada.
Com $\chi=1000$ o heatmap já está, na prática, nesse limite (correção relativa $\sim|\gamma+i(2\Delta-E)|/2\chi\sim10^{-3}$).

In [ ]:
def K_E(E, p=P):
    """Núcleo não linear K(E), E = energia total."""
    return 1.0 / (1.0 + 2j * p.chi / (p.gamma + 1j * (2 * p.Delta - E)))


E_teste = np.array([-2.0, 0.0, 1.0, 3.0])
limite = (P.gamma + 1j * (2 * P.Delta - E_teste)) / 2j
for chi in (1.0, 10.0, 1000.0, 1e6):
    rel = np.max(np.abs(chi * K_E(E_teste, Params(chi=chi)) - limite) / np.abs(limite))
    print(f"chi = {chi:>9g}   max |chi K - limite| / |limite| = {rel:.2e}")

---
## 4. A amplitude de saída $\psi_{\rm out}(\omega_a,\omega_b)$

### 4.1 Aplicando $S$ ao estado de entrada

$$
\psi_{\rm out}(\omega_a,\omega_b)=\iint d\nu_a\,d\nu_b\;S(\omega_a,\omega_b;\nu_a,\nu_b)\;\xi(\nu_a)\,\xi(\nu_b).
$$

* Na **parte linear**, as duas deltas $\delta(\omega_a-\nu_a)\delta(\omega_b-\nu_b)$ eliminam as duas integrais.
* Na **parte não linear**, a delta de energia elimina uma delas: $\nu_b=E-\nu_a$, com $E=\omega_a+\omega_b$.
  Chamando $\nu\equiv\nu_a$ e separando em $A^{N-j}B^{j-1}$ o que depende das frequências de saída
  ($a(\omega_a)^{N-j}a(\omega_b)^{j-1}$) do que depende das de entrada ($a(\nu_b)^{N-j}a(\nu_a)^{j-1}$):

$$
\boxed{\;
\psi_{\rm out}(\omega_a,\omega_b)=
\underbrace{\xi(\omega_a)\,\xi(\omega_b)\,S_1(\omega_a)\,S_1(\omega_b)}_{\text{linear}}
\;\underbrace{-\;\frac{i\chi\gamma^{2}}{\pi}\,\frac{K(E)}{\Gamma(\omega_a)\,\Gamma(\omega_b)}
\sum_{j=1}^{N}a(\omega_a)^{N-j}\,a(\omega_b)^{j-1}\,P_j(E)}_{\text{não linear}}\;}
$$

em que toda a dependência nas frequências de entrada ficou concentrada em

$$
\boxed{\;
P_j(E)=\int d\nu\;
\frac{\xi(\nu)\,a(\nu)^{j-1}}{\Gamma(\nu)}\;
\frac{\xi(E-\nu)\,a(E-\nu)^{N-j}}{\Gamma(E-\nu)}\;}
$$

### 4.2 $P_j$ é uma convolução

Definindo

$$
C_j(\nu)=\frac{\xi(\nu)\,a(\nu)^{j-1}}{\Gamma(\nu)},\qquad
D_j(\nu)=\frac{\xi(\nu)\,a(\nu)^{N-j}}{\Gamma(\nu)},
$$

tem-se

$$
P_j(E)=(C_j*D_j)(E)=\int d\nu\;C_j(\nu)\,D_j(E-\nu).
$$

Essa é a observação que torna o cálculo barato: em vez de uma integral por pixel ($n_\omega^2=4\times10^6$
integrais), bastam **$N$ convoluções 1D**, feitas por FFT (`fftconvolve`), cada uma dando $P_j$ numa grade de
energias. Depois, $P_j$ é **interpolado** em $E=\omega_a+\omega_b$ de todos os pixels de uma vez.

### 4.3 Implementação discreta (`psi_out`)

1. Grade uniforme $\nu_k$, $k=0,\dots,n-1$, com espaçamento $h$ (seção 6).
2. `base` $=\xi(\nu)/\Gamma(\nu)$ e `a` $=a(\nu)$ são pré-calculados.
3. A convolução discreta de dois vetores de $n$ pontos tem $2n-1$ pontos, nas energias
   $E_m=2\nu_0+m\,h$, $m=0,\dots,2n-2$ (`E_grid`). Com a regra do retângulo,
   $$
   P_j(E_m)\approx h\sum_k C_j(\nu_k)\,D_j(E_m-\nu_k)=h\,(C_j\ast D_j)_m .
   $$
4. `np.interp` interpola as partes real e imaginária de $P_j$ em $E=\omega_a+\omega_b$ (valor $0$ fora de `E_grid`).
5. Acumula-se $\sum_j a(\omega_a)^{N-j}a(\omega_b)^{j-1}P_j(E)$ em `soma`; multiplica-se por
   $-\dfrac{i\chi\gamma^2}{\pi}\dfrac{K(E)}{\Gamma(\omega_a)\Gamma(\omega_b)}$.
6. Se `aplicar_correcao`, as duas partes são multiplicadas por $S_1^*(\omega_a)S_1^*(\omega_b)$ (seção 5).

A função devolve `(total, nao_linear)`.

In [ ]:
def grade(p=P):
    """Grade de frequências para as convoluções P_j(E), e seu espaçamento (seção 6)."""
    half = p.k_sigma * p.sigma
    scales = [p.sigma, p.gamma]
    if p.L != 0:
        scales.append(1.0 / (p.N * abs(p.L)))
    n = int(2 * half / (min(scales) / p.pts_per_width)) + 1
    n = min(max(n, p.n_min), p.n_max) | 1
    w = np.linspace(p.omega_c - half, p.omega_c + half, n)
    return w, w[1] - w[0]


def psi_out(WA, WB, p=P):
    """Amplitude do par espalhado nas frequências de saída (WA, WB): (total, parte não linear)."""
    w, h = grade(p)
    base = xi_in(w, p) / Gamma(w, p)
    a = a_site(w, p)
    E_grid = np.linspace(2 * w[0], 2 * w[-1], 2 * len(w) - 1)

    E = WA + WB
    aA, aB = a_site(WA, p), a_site(WB, p)

    soma = np.zeros_like(E, dtype=complex)
    for j in range(1, p.N + 1):
        # P_j(E) = (C * D)(E), com C = xi a^{j-1}/Gamma e D = xi a^{N-j}/Gamma
        P_grid = fftconvolve(base * a ** (j - 1), base * a ** (p.N - j)) * h
        Pj = (np.interp(E, E_grid, P_grid.real, left=0.0, right=0.0)
              + 1j * np.interp(E, E_grid, P_grid.imag, left=0.0, right=0.0))
        soma += aA ** (p.N - j) * aB ** (j - 1) * Pj

    K = K_E(E, p)

    linear = xi_in(WA, p) * xi_in(WB, p) * S1(WA, p) * S1(WB, p)
    nao_linear = -(1j * p.chi * p.gamma ** 2 / np.pi) * K * soma / (Gamma(WA, p) * Gamma(WB, p))

    if p.aplicar_correcao:
        c = np.conj(S1(WA, p)) * np.conj(S1(WB, p))
        linear, nao_linear = linear * c, nao_linear * c
    return linear + nao_linear, nao_linear

---
## 5. Correção da fase de fóton único (`aplicar_correcao`)

A fase $S_1(\omega_a)S_1(\omega_b)$ é a que cada fóton ganharia **sozinho**; ela não diz nada sobre a interação,
mas oscila rapidamente em $\omega$ (principalmente para $N$ e $L$ grandes) e poluiria os mapas de fase.
Com `aplicar_correcao = True`, as duas partes são multiplicadas por

$$
\mathcal C(\omega_a,\omega_b)=S_1^*(\omega_a)\,S_1^*(\omega_b),
$$

o que equivale a descrever o estado de saída **em relação ao alvo linear** $S_1\times S_1$
(é a mesma função `correction(w)` de `fidelidade_sigma/calcular_sigma_n_sitios.py`). Então:

* a parte linear vira simplesmente $\xi(\omega_a)\,\xi(\omega_b)$, real e positiva;
* como $|S_1|=1$, **$|\psi_{\rm out}|$ e $|\psi^{\rm n\tilde ao\ linear}_{\rm out}|$ não mudam**: a correção só afeta
  `'real'`, `'imag'` e `'phase'`.

Nessa convenção, uma **porta de fase condicional ideal** (fase $\pi$) daria

$$
\psi_{\rm out}^{\rm ideal}(\omega_a,\omega_b)=-\,\xi(\omega_a)\,\xi(\omega_b),
$$

ou seja: o mesmo módulo do pacote de entrada, com o sinal trocado. O heatmap mostra o quanto o resultado real
se afasta disso.

In [ ]:
ax1 = np.linspace(-3, 3, 121)
WA, WB = np.meshgrid(ax1, ax1)
Pc = Params(N=4, L=0.5)
Ps = Params(N=4, L=0.5, aplicar_correcao=False)
tot_c, nl_c = psi_out(WA, WB, Pc)
tot_s, nl_s = psi_out(WA, WB, Ps)
print("|psi| igual com e sem correção      :", np.allclose(np.abs(tot_c), np.abs(tot_s)))
print("fase muda                            :", not np.allclose(np.angle(tot_c), np.angle(tot_s)))
lin_c = tot_c - nl_c
print("parte linear corrigida = ξ(wa)ξ(wb)  :", np.allclose(lin_c, xi_in(WA, Pc) * xi_in(WB, Pc)))

---
## 6. Grade numérica das convoluções (`grade`)

As convoluções $P_j$ são feitas numa grade uniforme $\nu\in[\omega_c-h_w,\;\omega_c+h_w]$.

**Janela.** As gaussianas $\xi(\nu)\,\xi(E-\nu)$ confinam o integrando a $|\nu-\omega_c|\lesssim k_\sigma\sigma$:

$$
h_w=k_\sigma\,\sigma,\qquad k_\sigma=9\quad\Rightarrow\quad \xi\ \text{cai por}\ e^{-k_\sigma^2/4}\approx1.6\times10^{-9}\ \text{na borda}.
$$

**Resolução.** O espaçamento resolve a **menor escala** do integrando:

* $\sigma$ — largura do pacote;
* $\gamma$ — largura das lorentzianas $1/\Gamma$ e da variação de fase de $t$;
* $1/(N|L|)$ — só se $L\neq0$: o termo $a(\nu)^{N-1}$ contém $e^{i(N-1)\nu L}$, que oscila com período
  $\sim2\pi/(NL)$ em $\nu$; quanto maior a cadeia, mais fina a grade precisa ser.

$$
n=\left\lfloor\frac{2h_w}{\min(\sigma,\gamma,1/(N|L|))/14}\right\rfloor+1,
\qquad n\in[\,2001,\;600001\,]\ \text{e forçado a ser ímpar (}\texttt{| 1}\text{)}
$$

O número ímpar de pontos coloca um ponto exatamente em $\omega_c$.

**Grade de energias.** `E_grid` cobre $E\in[2\omega_c-2h_w,\;2\omega_c+2h_w]$ e $P_j$ é posto em **zero** fora dela.
Portanto o plano do heatmap precisa satisfazer $|\omega_a+\omega_b-2\omega_c|<2k_\sigma\sigma$ — com $\sigma=1$ isso é
$|E|<18$, folgadamente satisfeito pela faixa $[-3,3]^2$.

In [ ]:
print(f"{'N':>3} {'L':>4} {'sigma':>6} {'n':>7} {'h':>10}  escala dominante")
for N_, L_, s_ in [(10, 0, 1.0), (10, 0, 0.1), (10, 0.5, 1.0), (50, 1.0, 1.0), (10, 0, 0.01)]:
    p = Params(N=N_, L=L_, sigma=s_)
    w, h = grade(p)
    escalas = {'sigma': s_, 'gamma': p.gamma}
    if L_:
        escalas['1/(NL)'] = 1 / (N_ * L_)
    dom = min(escalas, key=escalas.get)
    print(f"{N_:3d} {L_:4.1f} {s_:6.2f} {len(w):7d} {h:10.2e}  {dom}" + ("  (piso n_min)" if len(w) == p.n_min else ""))

---
## 7. Verificações

### 7.1 Convolução por FFT × integral direta

Para um ponto $(\omega_a,\omega_b)$ qualquer, calculamos a parte não linear pela fórmula da seção 4.1 com a
integral em $\nu$ feita por `scipy.integrate.quad`, e comparamos com `psi_out`. A pequena diferença vem da
regra do retângulo e da interpolação linear em $E$.

In [ ]:
def nao_linear_direto(wa, wb, p=P):
    """Parte não linear de psi_out num único ponto, com a integral em nu feita por quad."""
    E = wa + wb

    def integrando(nu, parte):
        s = sum(a_site(wa, p) ** (p.N - j) * a_site(wb, p) ** (j - 1)
                * xi_in(nu, p) * a_site(nu, p) ** (j - 1) * xi_in(E - nu, p) * a_site(E - nu, p) ** (p.N - j)
                / (Gamma(nu, p) * Gamma(E - nu, p))
                for j in range(1, p.N + 1))
        return s.real if parte == 're' else s.imag

    lim = (p.omega_c - p.k_sigma * p.sigma, p.omega_c + p.k_sigma * p.sigma)
    soma = (quad(integrando, *lim, args=('re',), limit=500)[0]
            + 1j * quad(integrando, *lim, args=('im',), limit=500)[0])
    nl = -(1j * p.chi * p.gamma ** 2 / np.pi) * K_E(E, p) * soma / (Gamma(wa, p) * Gamma(wb, p))
    if p.aplicar_correcao:
        nl *= np.conj(S1(wa, p)) * np.conj(S1(wb, p))
    return nl


print(f"{'N':>3} {'L':>4} {'(wa, wb)':>13} {'quad':>26} {'psi_out (FFT)':>26} {'dif. rel.':>10}")
for N_, L_, (wa, wb) in [(1, 0, (0.4, 0.2)), (2, 0.5, (0.3, -0.7)), (4, 0.5, (0.3, -0.7)), (10, 0, (1.2, 0.1))]:
    p = Params(N=N_, L=L_)
    d = nao_linear_direto(wa, wb, p)
    f = psi_out(np.array([wa]), np.array([wb]), p)[1][0]
    print(f"{N_:3d} {L_:4.1f} {str((wa, wb)):>13} {d:26.6f} {f:26.6f} {abs(f - d) / abs(d):10.1e}")

### 7.2 Unitariedade: $\iint|\psi_{\rm out}|^2=1$

A matriz S é unitária e a entrada é normalizada, então a saída também precisa ser:

$$
\iint d\omega_a\,d\omega_b\;|\psi_{\rm out}(\omega_a,\omega_b)|^{2}=\langle\xi\xi|S^\dagger S|\xi\xi\rangle=1 .
$$

Esse é um teste forte: a parte não linear sozinha **não** tem norma pequena, e a soma só dá 1 se o
termo linear e o não linear tiverem a interferência certa. Integramos numa janela $[-10,10]^2$
(a pequena falta vem das caudas $\propto1/|\Gamma(\omega_a)\Gamma(\omega_b)|^2$ fora da janela).

### 7.3 Ligação com a fidelidade $F_1$

A amplitude $F$ usada em `fidelidade_sigma/` é a **sobreposição** da saída (já corrigida) com a entrada:

$$
F=\langle\xi\xi|\,S_1^\dagger S_1^\dagger\,S\,|\xi\xi\rangle
=\iint d\omega_a\,d\omega_b\;\xi(\omega_a)\,\xi(\omega_b)\;\psi_{\rm out}(\omega_a,\omega_b)
\quad(\texttt{aplicar\_correcao=True}),
$$

e a fidelidade é $F_1(\pi)=\big[6-3\,\mathrm{Re}(F)+|F|^2\big]/10$. Ou seja, **$F$ é a projeção do heatmap
(complexo) sobre o pacote de entrada**: a porta ideal ($\psi_{\rm out}=-\xi\xi$) dá $F=-1$ e $F_1=1$.
Comparamos com os `.npz` gerados por `calcular_sigma_n_sitios.py` (quando existem).

In [ ]:
DATA_DIR = os.path.join('..', 'fidelidade_sigma', 'data', 'n_sitios')

ax2 = np.linspace(-10, 10, 801)
dw = ax2[1] - ax2[0]
WA2, WB2 = np.meshgrid(ax2, ax2)

print(f"{'N':>3} {'L':>4} {'norma':>9} {'F (heatmap)':>22} {'F (.npz)':>22} {'F1(pi) heatmap':>15}")
for N_, L_ in [(1, 0), (2, 0), (4, 0), (4, 0.4), (10, 0), (10, 1.0)]:
    p = Params(N=N_, L=L_)
    tot, _ = psi_out(WA2, WB2, p)
    norma = np.sum(np.abs(tot) ** 2) * dw * dw
    F = np.sum(xi_in(WA2, p) * xi_in(WB2, p) * tot) * dw * dw
    F1 = (6 - 3 * F.real + abs(F) ** 2) / 10

    F_npz = '—'
    arq = os.path.join(DATA_DIR, f'fidelity_N{N_}_L{L_:g}_chi1000_gamma1_omega0.npz')
    if os.path.exists(arq):
        d = np.load(arq)
        i = np.flatnonzero(np.isclose(d['sigmas'], p.sigma))
        if i.size:
            F_npz = f"{complex(d['F_re'][i[0]], d['F_im'][i[0]]):.5f}"
    print(f"{N_:3d} {L_:4.1f} {norma:9.5f} {F:22.5f} {F_npz:>22} {F1:15.4f}")

* A norma fica em $1$ dentro de $\sim10^{-3}$ para todos os $N$ e $L$.
* Onde há `.npz` com $\sigma=1$ exatamente, o $F$ obtido **integrando o heatmap** reproduz o $F$ do cálculo de
  fidelidade (que usa outra rota numérica: produto de convoluções integrado em $E$). As duas implementações
  descrevem o mesmo estado de saída.
* Para $N=1$ o resultado não depende de $L$: com um único sítio não há propagação entre sítios
  ($S_1=t$ e $a^{0}=1$ na soma).

---
## 8. O gráfico, elemento por elemento

O heatmap é uma grandeza real derivada de $\psi_{\rm out}(\omega_a,\omega_b)$ (por padrão, $|\psi_{\rm out}|$),
desenhada sobre a malha $\omega_a\times\omega_b$.

| Elemento | Código no script | Expressão / significado |
|---|---|---|
| Malha | `np.meshgrid(eixo, eixo, indexing='xy')` | `WA` varia ao longo das colunas ($x$), `WB` ao longo das linhas ($y$) |
| Eixo $x$ | `xlabel = r'$\omega_a$'` | frequência de saída do fóton $a$, em unidades de $\gamma$ |
| Eixo $y$ | `ylabel = r'$\omega_b$'` | frequência de saída do fóton $b$ |
| Faixa | `omega_min, omega_max = -3, 3` | a mesma nos dois eixos; `set_aspect('equal')` mantém a figura quadrada |
| Cor | `field='abs'` | $\lvert\psi_{\rm out}\rvert$ (seção 9 para os outros campos) |
| Rótulo da barra | `FIELD_LABELS[field]` | por exemplo $\lvert\psi_{\rm out}(\omega_a,\omega_b)\rvert$ |
| Faixa de cores | `vmin=0.0, vmax=None` | `vmin=0` prende o preto no zero (linhas nodais aparecem pretas); `vmax=None` usa o máximo dos dados |
| Título | gerado se `title=None` | $N,\ \chi,\ \gamma,\ \omega_0,\ L,\ \sigma$ |
| Desenho | `pcolormesh(..., shading='auto', rasterized=True)` | cada pixel é um retângulo; `rasterized` embute a malha como imagem no SVG (senão seriam milhões de polígonos) |
| Arquivo | `outfile = f'heatmap_omegaa_omegab_N{N}_l{L}_sigma_{sigma}.svg'` | formato deduzido da extensão, `dpi=300` para a parte rasterizada |
| Tamanho | `figsize=(7, 6)` | em polegadas |
| Janela | `show=True` | abre a janela interativa além de salvar |

### 8.1 O mapa de cores

`cores` é uma rampa **sequencial** preto → violeta → dourado claro:

```
['#000000', '#05050F', '#211546', '#562369', '#943A65', '#C75E50', '#E68F4D', '#F5C578', '#FDF6DC']
```

Os passos de **luminosidade** são uniformes (OKLab $L$ de $0$ a $0.99$, passo $0.122$). Como a informação está na
luminosidade, a cor codifica magnitude de forma **monótona**: o mapa continua legível em tons de cinza e para
daltônicos. `LinearSegmentedColormap.from_list` interpola as 9 cores em `n_niveis = 256` níveis.

| Variável | Valor | Efeito |
|---|---|---|
| `cores` | lista acima | as cores da rampa, do menor ao maior valor |
| `cores_invertidas` | `False` | `True` inverte (fundo claro, lóbulos escuros) |
| `n_niveis` | `256` | número de níveis interpolados |
| `cmap` | `None` | nome de um colormap do matplotlib (`'magma'`, `'cividis'`, `'twilight'` para `'phase'`); se definido, ignora `cores` |

In [ ]:
cores = ['#000000', '#05050F', '#211546', '#562369', '#943A65',
         '#C75E50', '#E68F4D', '#F5C578', '#FDF6DC']
MAPA = LinearSegmentedColormap.from_list('heatmap', cores, N=256)

FIELD_LABELS = {
    'abs': r'$|\psi_{\mathrm{out}}(\omega_a,\omega_b)|$',
    'abs_nl': r'$|\psi_{\mathrm{out}}^{\mathrm{n\tilde ao\ linear}}(\omega_a,\omega_b)|$',
    'real': r'$\mathrm{Re}\,\psi_{\mathrm{out}}(\omega_a,\omega_b)$',
    'imag': r'$\mathrm{Im}\,\psi_{\mathrm{out}}(\omega_a,\omega_b)$',
    'phase': r'$\arg\,\psi_{\mathrm{out}}(\omega_a,\omega_b)$',
}


def campo(total, nao_linear, field):
    return {'abs': np.abs(total), 'abs_nl': np.abs(nao_linear), 'real': np.real(total),
            'imag': np.imag(total), 'phase': np.angle(total)}[field]


def heatmap(p=P, field='abs', ax=None, cmap=MAPA, vmin=0.0, vmax=None, colorbar=True):
    """Reproduz main() do script para os parâmetros p."""
    eixo = np.linspace(p.omega_min, p.omega_max, p.n_omega)
    WA, WB = np.meshgrid(eixo, eixo, indexing='xy')     # WA no eixo x, WB no eixo y
    total, nao_linear = psi_out(WA, WB, p)
    z = campo(total, nao_linear, field)
    if ax is None:
        _, ax = plt.subplots(figsize=(7, 6))
    im = ax.pcolormesh(eixo, eixo, z, cmap=cmap, vmin=vmin, vmax=vmax, shading='auto', rasterized=True)
    if colorbar:
        ax.figure.colorbar(im, ax=ax, label=FIELD_LABELS[field])
    ax.set_xlabel(r'$\omega_a$')
    ax.set_ylabel(r'$\omega_b$')
    ax.set_aspect('equal')
    ax.set_title(rf'$N={p.N},\ \chi={p.chi:g},\ \gamma={p.gamma:g},\ \omega_0={p.omega0:g},\ '
                 rf'L={p.L:g},\ \sigma={p.sigma:g}$')
    return ax, z


# figura com os parâmetros do script (N=10, L=0, sigma=1)
ax, z = heatmap(P)
ax.figure.tight_layout()
plt.show()
print("máximo de |psi_out| =", z.max())

---
## 9. Os outros campos (`field`)

| `field` | Grandeza | Comentário |
|---|---|---|
| `'abs'` | $\lvert\psi_{\rm out}\rvert$ | onde está a amplitude de saída; zeros (linhas pretas) indicam interferência destrutiva entre as partes linear e não linear |
| `'abs_nl'` | $\lvert\psi_{\rm out}^{\text{não linear}}\rvert$ | só o segundo termo: onde a interação redistribui amplitude |
| `'real'` | $\mathrm{Re}\,\psi_{\rm out}$ | com a correção, a porta ideal daria $-\xi(\omega_a)\xi(\omega_b)$ (tudo negativo) |
| `'imag'` | $\mathrm{Im}\,\psi_{\rm out}$ | a porta ideal daria zero |
| `'phase'` | $\arg\psi_{\rm out}\in(-\pi,\pi]$ | a porta ideal daria $\pi$ em todo o plano |

**Cuidado com `vmin`.** O padrão `vmin = 0.0` é certo para `'abs'` e `'abs_nl'`, mas **corta os valores negativos**
de `'real'`, `'imag'` e `'phase'`. Para esses, use `vmin=None` ou uma faixa simétrica, e de preferência um
colormap divergente ou cíclico (`'twilight'` para a fase), como abaixo.

In [ ]:
p9 = Params(N=4, L=0.0, n_omega=300)
fig, axs = plt.subplots(2, 2, figsize=(11, 9.5))
heatmap(p9, 'abs', axs[0, 0])
heatmap(p9, 'abs_nl', axs[0, 1])
_, zr = heatmap(p9, 'real', axs[1, 0], cmap='RdBu_r', vmin=-0.4, vmax=0.4)
heatmap(p9, 'phase', axs[1, 1], cmap='twilight', vmin=-np.pi, vmax=np.pi)
fig.tight_layout()
plt.show()

---
## 10. Dependência em $N$ e em $L$

As figuras salvas nesta pasta (`heatmap_omegaa_omegab_N{1,2,4,10}_l0_sigma_1.0.svg`) usam
$\chi=1000$, $\gamma=1$, $\omega_0=0$, $L=0$, $\sigma=1$. Abaixo elas são refeitas (com menos pixels).

O que se observa:

* **Simetria de troca.** $\psi_{\rm out}(\omega_a,\omega_b)=\psi_{\rm out}(\omega_b,\omega_a)$ para quaisquer $N$ e $L$:
  a troca $\omega_a\leftrightarrow\omega_b$ leva o termo $j$ da soma no termo $N+1-j$, e $P_{N+1-j}(E)=P_j(E)$
  (basta trocar $\nu\to E-\nu$ na integral). Os mapas são sempre simétricos em relação à diagonal $\omega_a=\omega_b$
  (checado numericamente na primeira célula abaixo).
* **$N=1$:** um único lóbulo centrado na origem, alongado ao longo da diagonal $\omega_a=\omega_b$, com
  máximo $\approx0.71$ — bem acima de $0.40$, o máximo do pacote de entrada: a interação concentra amplitude perto
  da ressonância.
* **$N=2$:** surgem linhas nodais (pretas) perto de $\omega_a\approx0$ e $\omega_b\approx0$, que dividem a
  amplitude em quatro lóbulos.
* **$N=4$ e $N=10$:** as linhas nodais se multiplicam em franjas hiperbólicas. Cada termo $j$ da soma carrega uma
  fase diferente, $a(\omega_a)^{N-j}a(\omega_b)^{j-1}$, e os $N$ termos interferem; quanto maior $N$, mais rápido
  essas fases variam com $\omega$ e mais franjas aparecem.
* **$L\neq0$** acrescenta as fases $e^{i\omega L}$ em cada potência de $a$, o que desloca e deforma as linhas
  nodais (última figura: $N=4$ com $L=0$, $0.5$ e $1$).

Em todos os casos, a amplitude **se espalha para fora** do pacote de entrada: o estado de saída não é
$-\xi(\omega_a)\xi(\omega_b)$. Essa redistribuição espectral é o que faz $|F|<1$ (seção 7.3) e a fidelidade
$F_1$ ficar abaixo de 1.

In [ ]:
# simetria de troca wa <-> wb (a malha é quadrada, então trocar os eixos é transpor)
ax3 = np.linspace(-3, 3, 201)
WA3, WB3 = np.meshgrid(ax3, ax3)
for N_, L_ in [(2, 0), (4, 0.5), (10, 1.0)]:
    tot, _ = psi_out(WA3, WB3, Params(N=N_, L=L_))
    print(f"N={N_:2d}  L={L_:3.1f}   psi(wa,wb) = psi(wb,wa): {np.allclose(tot, tot.T, atol=1e-8)}")

In [ ]:
fig, axs = plt.subplots(2, 2, figsize=(11, 9.5))
for ax, N_ in zip(axs.flat, (1, 2, 4, 10)):
    _, z = heatmap(Params(N=N_, n_omega=300), 'abs', ax)
    print(f"N={N_:2d}   max |psi_out| = {z.max():.3f}")
fig.tight_layout()
plt.show()

In [ ]:
fig, axs = plt.subplots(1, 3, figsize=(16, 5))
for ax, L_ in zip(axs, (0.0, 0.5, 1.0)):
    heatmap(Params(N=4, L=L_, n_omega=300), 'abs', ax)
fig.tight_layout()
plt.show()